# 정제된 Prophet 파이프라인

`04_original_prophet_submission.ipynb` 의 흐름을 단계별로 정리한 self-contained 노트북입니다. 건물 단위로 Prophet 모델을 학습하고, 학습이 불안정한 건물에 대해서는 요일·시간대 중앙값 fallback 으로 대체합니다.


## Imports & Setup

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from prophet import Prophet

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data")
OUTPUT_PATH = Path("submissions/prophet_submission.csv")

ID_COLUMN = "num_date_time"
BUILDING_COLUMN = "building_id"
TIMESTAMP_COLUMN = "timestamp"
TARGET_COLUMN = "power_kwh"
ANSWER_COLUMN = "answer"

REGRESSORS = ("temperature", "rainfall", "wind_speed", "humidity",
              "hour_sin", "hour_cos", "day_of_week", "is_weekend")
WEATHER_COLUMNS = ("temperature", "rainfall", "wind_speed", "humidity")
MIN_PREDICTION = 0.0
FALLBACK_RECENT_HOURS = 24 * 14

## 데이터 로드

대회 컬럼명을 안정적인 영문 식별자로 정규화합니다.


In [ ]:
COLUMN_RENAME = {
    "num_date_time": ID_COLUMN,
    "건물번호": BUILDING_COLUMN,
    "일시": TIMESTAMP_COLUMN,
    "기온(C)": "temperature",
    "강수량(mm)": "rainfall",
    "풍속(m/s)": "wind_speed",
    "습도(%)": "humidity",
    "일조(hr)": "sunshine",
    "일사(MJ/m2)": "solar_radiation",
    "전력소비량(kWh)": TARGET_COLUMN,
}


def normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.rename(columns={k: v for k, v in COLUMN_RENAME.items() if k in frame.columns})
    if TIMESTAMP_COLUMN in out.columns:
        out[TIMESTAMP_COLUMN] = pd.to_datetime(out[TIMESTAMP_COLUMN])
    return out


train = normalize_columns(pd.read_csv(DATA_DIR / "train.csv"))
test = normalize_columns(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"train: {train.shape}, test: {test.shape}, submission: {sample_submission.shape}")

## 결측치 보간 + 캘린더 파생 피처

In [ ]:
def fill_weather(series: pd.Series) -> pd.Series:
    filled = series.interpolate(limit_direction="both")
    filled = filled.ffill().bfill()
    return filled.fillna(0)


def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    ts = out[TIMESTAMP_COLUMN]
    out["hour"] = ts.dt.hour
    out["day_of_week"] = ts.dt.dayofweek
    out["month"] = ts.dt.month
    out["week_of_year"] = ts.dt.isocalendar().week.astype(int)
    out["is_weekend"] = out["day_of_week"].isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    return out


def prepare_frame(frame: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    out = frame.copy()
    out = out.sort_values([BUILDING_COLUMN, TIMESTAMP_COLUMN]).reset_index(drop=False)
    out = out.rename(columns={"index": "_original_index"})

    for col in WEATHER_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
            out[col] = out.groupby(BUILDING_COLUMN)[col].transform(fill_weather)

    if is_train and TARGET_COLUMN in out.columns:
        out[TARGET_COLUMN] = pd.to_numeric(out[TARGET_COLUMN], errors="coerce")
        out = out.dropna(subset=[TARGET_COLUMN])
        out = out[out[TARGET_COLUMN] >= 0].reset_index(drop=True)

    return add_calendar_features(out)


train_prepared = prepare_frame(train, is_train=True)
test_prepared = prepare_frame(test, is_train=False)
print(f"train_prepared: {train_prepared.shape}, test_prepared: {test_prepared.shape}")

## 건물별 Prophet 학습 함수

학습이 실패하거나 분산이 너무 작은 regressor 가 있으면 자동으로 요일·시간대 중앙값 fallback 으로 대체합니다.


In [ ]:
def select_stable_regressors(building_train: pd.DataFrame) -> list[str]:
    regressors = [c for c in REGRESSORS if c in building_train.columns]
    return [c for c in regressors if building_train[c].nunique(dropna=True) > 1]


def to_prophet_frame(building_train: pd.DataFrame, regressors: list[str]) -> pd.DataFrame:
    columns = [TIMESTAMP_COLUMN, TARGET_COLUMN, *regressors]
    out = building_train.loc[:, columns].rename(columns={
        TIMESTAMP_COLUMN: "ds",
        TARGET_COLUMN: "y",
    })
    return out.dropna(subset=["ds", "y"])


def to_future_frame(building_test: pd.DataFrame, regressors: list[str]) -> pd.DataFrame:
    columns = [TIMESTAMP_COLUMN, *regressors]
    return building_test.loc[:, columns].rename(columns={TIMESTAMP_COLUMN: "ds"})


def predict_with_prophet(building_train: pd.DataFrame, building_test: pd.DataFrame) -> pd.Series:
    regressors = select_stable_regressors(building_train)
    prophet_frame = to_prophet_frame(building_train, regressors)
    future_frame = to_future_frame(building_test, regressors)

    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=False,
        seasonality_mode="multiplicative",
        interval_width=0.8,
    )
    for regressor in regressors:
        model.add_regressor(regressor)

    model.fit(prophet_frame)
    forecast = model.predict(future_frame)
    predictions = forecast["yhat"].clip(lower=MIN_PREDICTION)
    return pd.Series(predictions.to_numpy(), index=building_test.index)


def predict_with_seasonal_median(building_train: pd.DataFrame, building_test: pd.DataFrame) -> pd.Series:
    history = building_train.tail(FALLBACK_RECENT_HOURS).copy()
    grouped = history.groupby(["day_of_week", "hour"])[TARGET_COLUMN].median()
    fallback = float(history[TARGET_COLUMN].median())

    values = []
    for _, row in building_test.iterrows():
        values.append(float(grouped.get((row["day_of_week"], row["hour"]), fallback)))
    return pd.Series(np.maximum(values, MIN_PREDICTION), index=building_test.index)


def predict_building(building_train: pd.DataFrame, building_test: pd.DataFrame) -> pd.Series:
    if building_train.empty:
        return pd.Series(np.zeros(len(building_test)), index=building_test.index)
    try:
        return predict_with_prophet(building_train, building_test)
    except Exception:
        return predict_with_seasonal_median(building_train, building_test)

## 건물 단위 예측 루프

In [ ]:
predictions = pd.Series(index=test_prepared.index, dtype=float)

for building_id in sorted(test_prepared[BUILDING_COLUMN].unique()):
    train_b = train_prepared[train_prepared[BUILDING_COLUMN] == building_id]
    test_b = test_prepared[test_prepared[BUILDING_COLUMN] == building_id]
    predictions.loc[test_b.index] = predict_building(train_b, test_b)

print(f"predictions ready — count={predictions.notna().sum()}")

## 원래 순서 복구 후 제출 파일 생성

In [ ]:
test_ordered = test_prepared.sort_values("_original_index")
prediction_ordered = predictions.loc[test_ordered.index].to_numpy()

submission = sample_submission.copy()
assert len(submission) == len(prediction_ordered), "submission length mismatch"
submission[ANSWER_COLUMN] = prediction_ordered

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"saved: {OUTPUT_PATH}")
submission.head()

## 정리

- 한국어 컬럼명을 영문 식별자로 통일하고 기상 변수 결측을 건물 단위 보간 + ffill/bfill 로 채웠습니다.
- `hour_sin/cos`, `day_of_week`, `is_weekend` 등 캘린더 파생 피처를 Prophet regressor 로 추가했습니다.
- 건물별 Prophet (multiplicative seasonality + daily/weekly) 모델을 학습하고, 학습이 불안정한 케이스에 대해서는 요일·시간대 중앙값 fallback 으로 대체했습니다.
- 처리된 결과를 `_original_index` 기준으로 원래 순서로 복구해 sample submission 형식으로 저장했습니다.
